### Extract & Load
Extract data from Volume and load into bronze layer


In [0]:
import sqlite3
import pandas as pd
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, DoubleType, BooleanType, TimestampType
)

CATALOG = "northwind_raw_data"
SCHEMA  = "source_sql_db"

conn = sqlite3.connect("/Volumes/northwind_raw_data/source_sql_db/raw_files/Northwind_db.sqlite")

tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")


def sqlite_type_to_spark(sqlite_type: str):
    """Converts SQLite types to Spark types."""
    t = sqlite_type.upper()
    if any(x in t for x in ["INT"]):
        return LongType()
    if any(x in t for x in ["REAL", "FLOAT", "DOUBLE", "NUMERIC", "DECIMAL"]):
        return DoubleType()
    if any(x in t for x in ["BOOL"]):
        return BooleanType()
    if any(x in t for x in ["DATE", "TIME"]):
        return TimestampType()
    return StringType()  # TEXT, CHAR, VARCHAR, BLOB and unknown types → String


def get_sqlite_schema(conn, table: str) -> StructType:
    """Reads the table schema via PRAGMA and returns a Spark StructType."""
    pragma = pd.read_sql(f"PRAGMA table_info([{table}])", conn)
    fields = [
        StructField(row["name"], sqlite_type_to_spark(row["type"]), nullable=True)
        for _, row in pragma.iterrows()
    ]
    return StructType(fields)


for table in tables["name"]:
    df_pandas = pd.read_sql(f"SELECT * FROM [{table}]", conn)
    schema    = get_sqlite_schema(conn, table)

    if df_pandas.empty:
        # Creates an empty Spark DataFrame with explicit schema
        df_spark = spark.createDataFrame([], schema)
        print(f"  ⚠️  {table} created empty (schema preserved)")
    else:
        df_spark = spark.createDataFrame(df_pandas, schema=schema)
        print(f"  ✅ {table} ingested")

    df_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}")

conn.close()
print("Ingestion complete!")